# KyTea の単語分割モデルを学習して分野に合わせる（辞書・部分的アノテーション）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/03_japanese_tokenization/kytea_train.ipynb)

NLP クックブック 03-23 ／ 日本語の形態素解析と文分割

医療のような専門分野の文章で、薬剤名や交ぜ書きの用語（「せん妄」「じょく瘡」）の分割を誤る。人手で分割したコーパスから単語分割のモデルを自分で学習し、分野の語の辞書と、誤りやすい箇所だけに境界を付けたデータを加えて直したい。KyTea の学習ツール train-kytea を使う。

In [1]:
!pip install -q kytea==0.2.0 conllu==6.1.1
# pip の kytea には学習ツールが入っていないので、KyTea 0.4.7 のリリース（約 39MB。学習済みの日本語モデルも入っている）をビルドする。
# リリースは configure 済みなので autoreconf も apt-get も要らない。--disable-shared で静的にリンクし、make install せずに使う。
# Colab（2 コア）では、ダウンロードとビルドで 3 分ほどの見込み
!wget -q https://www.phontron.com/kytea/download/kytea-0.4.7.tar.gz && tar xzf kytea-0.4.7.tar.gz
# ビルドのログは make.log に書き、失敗したときだけ末尾を表示する
!cd kytea-0.4.7 && ./configure -q --disable-shared > /dev/null && (make -j2 > make.log 2>&1 && echo "ビルド完了" || tail -20 make.log)
!kytea-0.4.7/src/bin/train-kytea --help 2>&1 | grep -E "^  -(tok|part|dict|notags) "

In [2]:
import subprocess
import urllib.request
import Mykytea
from conllu import parse

TRAIN = "kytea-0.4.7/src/bin/train-kytea"

# UD Japanese-GSD (CC BY-SA 4.0、https://github.com/UniversalDependencies/UD_Japanese-GSD)。人手で短単位に分割されている
# 数値を再現できるように、リリース r2.19 のタグに固定する（master は UD のリリースごとに変わる）
base = "https://raw.githubusercontent.com/UniversalDependencies/UD_Japanese-GSD/r2.19/ja_gsd-ud-"
def load(split):
    sents = parse(urllib.request.urlopen(base + split + ".conllu").read().decode("utf-8"))
    return [[t["form"] for t in s if isinstance(t["id"], int)] for s in sents]   # 複数語トークンなどの行を除く

def ok(w):
    return not set(w) & set("/& \\")   # KyTea の区切り記号（タグの "/"、候補の "&"）や空白を含む語は使わない

train, test = load("train"), load("test")
train = [s for s in train if all(map(ok, s))]
print(len(train), "文", sum(map(len, train)), "語（学習）/", len(test), "文（評価）")

# 学習データは 1 行 1 文で、単語を空白で区切る（-tok の形式。品詞や読みは付けない）
with open("gsd.tok", "w", encoding="utf-8") as f:
    f.writelines(" ".join(s) + "\n" for s in train)

def train_kytea(model, *args):
    """train-kytea で単語分割だけのモデルを学習する（-notags: 品詞・読みは学習しない）"""
    r = subprocess.run([TRAIN, "-tok", "gsd.tok", *args, "-notags", "-model", model], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr
    return Mykytea.Mykytea(f"-model {model} -notags")

base_model = train_kytea("gsd.model")   # データのダウンロードと学習で、Colab で 2 分ほどの見込み
default = Mykytea.Mykytea("-model kytea-0.4.7/data/model.bin -notags")   # KyTea に同梱のモデル

7011 文 167032 語（学習）/ 543 文（評価）


In [3]:
def spans(words):
    out, p = set(), 0
    for w in words:
        out.add((p, p + len(w))); p += len(w)
    return out

def word_f1(gold_sents, pred_sents):
    """単語単位の F1: 開始位置と終了位置が両方一致した単語だけを正解とする"""
    tp = n_gold = n_pred = 0
    for gold, pred in zip(gold_sents, pred_sents):
        g, p = spans(gold), spans(pred)
        tp += len(g & p); n_gold += len(g); n_pred += len(p)
    prec, rec = tp / n_pred, tp / n_gold
    return round(2 * prec * rec / (prec + rec), 4)

def segment_all(kytea, sents):
    return [list(kytea.getWS("".join(s))) for s in sents]

print("GSD の評価データの単語 F1")
print("  同梱のモデル  :", word_f1(test, segment_all(default, test)))
print("  GSD で学習    :", word_f1(test, segment_all(base_model, test)))
for s in ["新しい産業構造を作らなければいけない。", "しばらくの完全休養とリハビリが必要とのことでした。"]:
    print("同梱:", " ".join(default.getWS(s)))
    print("GSD :", " ".join(base_model.getWS(s)))

GSD の評価データの単語 F1
  同梱のモデル  : 0.8333
  GSD で学習    : 0.9715
同梱: 新し い 産業 構造 を 作 ら な けれ ば いけ な い 。
GSD : 新しい 産業 構造 を 作ら なけれ ば いけ ない 。
同梱: しばらく の 完全 休養 と リハビリ が 必要 と の こと で し た 。
GSD : しばらく の 完全 休養 と リハビリ が 必要 と の こと でし た 。


In [4]:
import re

# 分野の語の一覧（ここでは医療の語。実務では用語集や薬剤の一覧から作る）
terms = """せん妄 じょく瘡 誤えん 膿瘍 てんかん 嚥下 褥瘡 喀痰 耐性 術後
アセトアミノフェン ロキソプロフェン バラシクロビル レベチラセタム セフトリアキソン クロピドグレル
ビソプロロール ファモチジン メトホルミン プレドニゾロン サルブタモール レボフロキサシン アモキシシリン""".split()

# 分野の文（分割していない生の文）。部分的アノテーションでは、用語の前後と内部の境界だけを付ける
domain_raw = """高齢者はせん妄を起こしやすい。
せん妄の原因として脱水が考えられる。
じょく瘡の深さを観察する。
寝たきりの患者はじょく瘡ができやすい。
誤えんを防ぐために姿勢を工夫する。
誤えんの有無を確認した。
てんかんの診断で紹介された。
肝膿瘍の治療は長期に及ぶ。
ロキソプロフェンを食後に服用する。
アセトアミノフェンは解熱にも使われる。
バラシクロビルの内服を終了した。
セフトリアキソンの投与を終了した。
クロピドグレルの内服を再開した。
ビソプロロールで脈拍が落ち着いた。
喀痰の量が減った。
耐性菌の対策を強化する。
嚥下の訓練を続けている。""".split()

# 評価用の分野の文（上とは別の文。人手で短単位に分割した正解）
domain_test = [s.split() for s in """夜間 に せん妄 が 見 られ た ため 、 家族 に 付き添い を 依頼 し た 。
術後 せん妄 の 有無 を 毎日 評価 する 。
じょく瘡 の 予防 に 体位 変換 を 行う 。
仙骨 部 の じょく瘡 は 改善 し て いる 。
誤えん の リスク が 高い ため 、 食事 形態 を 変更 し た 。
誤えん 性 肺炎 で 入院 し た 。
てんかん 発作 が 起き た とき の 対応 を 説明 し た 。
肝 膿瘍 に 対し て 抗菌 薬 を 投与 し た 。
腹部 エコー で 膿瘍 を 認め た 。
発熱 に は アセトアミノフェン を 使用 する 。
腰痛 に 対し て ロキソプロフェン を 処方 し た 。
ロキソプロフェン で 胃 の 不快 感 が 出 た 。
バラシクロビル を 腎 機能 に 合わせ て 減量 する 。
レベチラセタム の 内服 を 継続 する 。
尿路 感染 症 に セフトリアキソン を 点滴 する 。
手術 の 前 に クロピドグレル を 中止 し た 。
脈拍 が 速い ため ビソプロロール を 増量 し た 。
ファモチジン は 腎 機能 が 低い と 減量 が 必要 だ 。
血糖 の 管理 に メトホルミン を 用いる 。
プレドニゾロン の 副 作用 に 注意 する 。
喀痰 から 耐性 菌 が 検出 さ れ た 。
嚥下 の 評価 を 行い 、 食事 を 再開 し た 。
褥瘡 の 状態 を 毎週 記録 する 。
吸入 に は サルブタモール を 用いる 。
レボフロキサシン と アモキシシリン の 併用 は 避ける 。""".split("\n")]

def partial_annotation(sent, terms):
    """文字の間に "|"（境界あり）、"-"（境界なし）、" "（不明）を挟む。用語の出現箇所だけ境界を決める"""
    b = [" "] * (len(sent) - 1)          # b[i] は sent[i] と sent[i+1] の間
    for t in terms:
        for m in re.finditer(re.escape(t), sent):
            i, j = m.start(), m.end()
            if i > 0: b[i - 1] = "|"
            if j < len(sent): b[j - 1] = "|"
            for k in range(i, j - 1): b[k] = "-"
    return "".join(c + x for c, x in zip(sent, b)) + sent[-1]

with open("domain.part", "w", encoding="utf-8") as f:
    f.writelines(partial_annotation(s, terms) + "\n" for s in domain_raw)
print(open("domain.part", encoding="utf-8").read()[:120])
print("評価:", len(domain_test), "文", sum(map(len, domain_test)), "語（うち分野の語",
      sum(w in terms for s in domain_test for w in s), "回）")

高 齢 者 は|せ-ん-妄|を 起 こ し や す い 。
せ-ん-妄|の 原 因 と し て 脱 水 が 考 え ら れ る 。
じ-ょ-く-瘡|の 深 さ を 観 察 す る 。
寝 た き り の 患 者 は|じ-ょ-く-瘡|が で 
評価: 25 文 247 語（うち分野の語 28 回）


In [5]:
from collections import Counter

# 辞書は 1 行 1 語（"語/読み" の形で読みも書ける）。3 通り作る
freq = Counter(w for s in train for w in s)
dicts = {
    "terms.dict": terms,                                           # 分野の語だけ
    "all+terms.dict": [w for w in freq if ok(w)] + terms,          # GSD の学習データの全語彙 + 分野の語
    "freq2+terms.dict": [w for w, n in freq.items() if n >= 2 and ok(w)] + terms,   # 2 回以上出た語 + 分野の語
}
for name, words in dicts.items():
    with open(name, "w", encoding="utf-8") as f:
        f.writelines(w + "\n" for w in dict.fromkeys(words))
    print(name, len(set(words)), "語")

def term_acc(gold_sents, pred_sents):
    """正解の中の分野の語のうち、ちょうど 1 語として切り出せた割合"""
    hit = n = 0
    for gold, pred in zip(gold_sents, pred_sents):
        p, pos = spans(pred), 0
        for w in gold:
            if w in terms:
                n += 1; hit += (pos, pos + len(w)) in p
            pos += len(w)
    return f"{hit}/{n}"

terms.dict 23 語
all+terms.dict 20101 語
freq2+terms.dict 8697 語


In [6]:
import pandas as pd

# 学習の設定ごとにモデルを作って比べる（Colab で全部で 4 分ほどの見込み）
settings = {
    "GSD のみ": [],
    "+ 辞書（分野の語だけ）": ["-dict", "terms.dict"],
    "+ 辞書（全語彙 + 分野の語）": ["-dict", "all+terms.dict"],
    "+ 辞書（2 回以上の語 + 分野の語）": ["-dict", "freq2+terms.dict"],
    "+ 部分的アノテーション": ["-part", "domain.part"],
    "+ 辞書（2 回以上の語 + 分野の語）+ 部分的アノテーション": ["-dict", "freq2+terms.dict", "-part", "domain.part"],
}
rows, models = [], {}
for i, (name, args) in enumerate(settings.items()):
    if args:
        k = train_kytea(f"model{i}.bin", *args)
    else:
        k = base_model   # 「GSD のみ」は学習済みのモデルを使い回す
    models[name] = k
    pred = segment_all(k, domain_test)
    rows.append({"設定": name, "分野の F1": word_f1(domain_test, pred), "分野の語": term_acc(domain_test, pred),
                 "誤った文": sum(p != g for p, g in zip(pred, domain_test)),
                 "GSD の F1": word_f1(test, segment_all(k, test))})
pd.DataFrame(rows).set_index("設定")

,分野の F1,分野の語,誤った文,GSD の F1
設定,,,,
GSD のみ,0.9249,17/28,11,0.9715
+ 辞書（分野の語だけ）,0.9249,17/28,11,0.9715
+ 辞書（全語彙 + 分野の語）,0.9405,28/28,10,0.9048
+ 辞書（2 回以上の語 + 分野の語）,0.9879,27/28,2,0.9748
+ 部分的アノテーション,0.9759,25/28,4,0.9714
+ 辞書（2 回以上の語 + 分野の語）+ 部分的アノテーション,0.9939,28/28,1,0.9748


In [7]:
# 誤りのあった文を、GSD だけのモデルと最後のモデルで見比べる
before, after = models["GSD のみ"], models[list(settings)[-1]]
for gold in domain_test:
    s = "".join(gold)
    b, a = list(before.getWS(s)), list(after.getWS(s))
    if b != gold or a != gold:
        print("GSD のみ:", " ".join(b))
        print("適応後  :", " ".join(a), "" if a == gold else "  ← まだ誤り（正解: " + " ".join(gold) + "）")

# 学習データの全語彙を辞書にしたモデルの出力
for s in ["じょく瘡の予防に体位変換を行う。", "腹部エコーで膿瘍を認めた。", "新しい産業構造を作らなければいけない。"]:
    print("全語彙の辞書:", " ".join(models["+ 辞書（全語彙 + 分野の語）"].getWS(s)))

GSD のみ: 夜間 に せん 妄 が 見 られ た ため 、 家族 に 付き添い を 依頼 し た 。
適応後  : 夜間 に せん妄 が 見 られ た ため 、 家族 に 付き添い を 依頼 し た 。 
GSD のみ: 術 後 せ ん 妄 の 有無 を 毎日 評価 する 。
適応後  : 術後 せん妄 の 有無 を 毎日 評価 する 。 
GSD のみ: じょく 瘡 の 予防 に 体位 変換 を 行う 。
適応後  : じょく瘡 の 予防 に 体位 変換 を 行う 。 
GSD のみ: 仙骨 部 の じょく 瘡 は 改善 し て いる 。
適応後  : 仙骨 部 の じょく瘡 は 改善 し て いる 。 
GSD のみ: 誤えん 性 肺 炎 で 入院 し た 。
適応後  : 誤えん 性 肺 炎 で 入院 し た 。   ← まだ誤り（正解: 誤えん 性 肺炎 で 入院 し た 。）
GSD のみ: 肝膿 瘍 に 対し て 抗菌 薬 を 投与 し た 。
適応後  : 肝 膿瘍 に 対し て 抗菌 薬 を 投与 し た 。 
GSD のみ: 発熱 に は アセトアミノ フェン を 使用 する 。
適応後  : 発熱 に は アセトアミノフェン を 使用 する 。 
GSD のみ: 腰痛 に 対し て ロキソ プロフェン を 処方 し た 。
適応後  : 腰痛 に 対し て ロキソプロフェン を 処方 し た 。 
GSD のみ: ロキソ プロフェン で 胃 の 不快 感 が 出 た 。
適応後  : ロキソプロフェン で 胃 の 不快 感 が 出 た 。 
GSD のみ: 尿路 感染 症 に セフトリア キソン を 点滴 する 。
適応後  : 尿路 感染 症 に セフトリアキソン を 点滴 する 。 
GSD のみ: 吸入 に は サルブ タモール を 用いる 。
適応後  : 吸入 に は サルブタモール を 用いる 。 
全語彙の辞書: じょく瘡 の 予防 に 体 位 変換 を 行う 。
全語彙の辞書: 腹部 エコ ー で 膿瘍 を 認め た 。
全語彙の辞書: 新しい 産業 構造 を 作ら なけれ ば いけ ない 。


- 同梱のモデルは GSD の評価データで単語 F1 が 0.8333、GSD で学習したモデルは 0.9715 だった。同梱のモデルは活用語尾を切り離す（「新し い」「作 ら な けれ ば」「で し た」）ので、この差には精度だけでなく単位の基準の違いが大きく効いている。評価データや後段の処理と同じ基準で分割したコーパスで学習する。GSD は BCCWJ の短単位（数詞だけはまとめる）で人手で分割されている。
- この章の「KyTea で単語分割と読みの推定をする」の注意書きのとおり KyTea は点予測なので、一部の境界だけを決めた部分的アノテーションでも学習できる（付録「仕組みを理解するための実装」の「単語分割を文字単位の系列ラベリングとして解く」の CRF では、すべての文字にラベルの付いた文が要る）。ここでは分野の文 17 文の用語の前後と内部だけに境界を付けて加え、評価の 25 文で分野の語を正しく切り出せた数が 17/28 から 25/28 になり、GSD の F1 はほぼ変わらなかった（0.9715 と 0.9714）。
- 辞書は、学習データの中で辞書の語と一致した箇所から辞書の素性の重みを学習する（KyTea のソースの `wsDictionaryFeatures` は `-dict` で与えた語だけを使う）。分野の語だけの辞書は GSD の文にほとんど現れないので、結果は辞書なしと同じになった。学習データの全語彙を辞書に入れると、分野の語は 28/28 になったが GSD の F1 が 0.9048 に下がり、辞書にない語を辞書にある短い語に切った（「体 位」「エコ ー」）。学習データでは正解の語が必ず辞書にあるので、辞書を信じすぎるようになったと考えられる。2 回以上出た語に絞ると、分野の語 27/28、GSD の F1 0.9748 になり、分野の語をほぼ切り出しつつ GSD の F1 も辞書なしより少し高かった。学習データとは別に作られた、同じ単位の一般の辞書に分野の語を足す方法も考えられるが、ここでは試していない。
- 辞書と部分的アノテーションを両方使うと、分野の語は 28/28、誤りのある文は 25 文中 1 文になった。残った誤りの「肺 炎」は、GSD の学習データに一度も出てこず、分野の語の一覧にも入れていない語。評価の文と用語の一覧はこのレシピのために作った小さなもの（25 文、247 語）で、評価に出る用語をすべて辞書に入れているので、結果は楽観的。設定の間の 1〜2 語の差は、このデータ量ではばらつきの範囲と考える。
- リリースの tar.gz は configure 済みなので、構文解析の章の「Ckylark で日本語の句構造解析をする」（GitHub のソースから autoreconf でビルドする）と違って apt-get が要らない。`--disable-shared` で静的にリンクすれば、`make install` や `ldconfig` をせずに `src/bin/train-kytea` を使える。学習したモデルは pip の `kytea` (0.2.0) でそのまま読み込める。tar.gz の `data/model.bin` は、この章の「KyTea で単語分割と読みの推定をする」でダウンロードするモデルと同じファイル（展開後のハッシュが一致した）。
- 検証では、何度実行しても表の数値は同じだった（GSD は r2.19。master も同じ内容だった）。KyTea のソースは Apache License 2.0、同梱のモデルの利用条件は「KyTea で単語分割と読みの推定をする」の注意書きを参照。GSD は CC BY-SA 4.0 なので、学習したモデルを配布するときは表示と継承の条件の扱いを確かめる。用語の一覧と医療の文はこのレシピのために作った架空の例。